In [1]:
import torch, gc
import random

from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from model_utils import load_transformer_model, load_silence_latent, load_encoder, decode_latent_and_save_audio, \
    get_files_in_path_as_array, load_finetuning_audio_segments, encode_audio_segments

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
model_dtype = torch.bfloat16
model_repo = "ACE-Step/acestep-v15-turbo-shift1"
checkpoint_dir = './checkpoints/'
checkpoint_file = checkpoint_dir + 'checkpoint.pt'

cuda


In [3]:
# from model_utils import get_reference_audio_latent
#
# vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device, model_dtype)
# path = "inputs/lora"
# files = get_files_in_path_as_array(path)
# ref = get_reference_audio_latent('./inputs/lora/pop.wav', vae, device, model_dtype, None)[0].transpose(1, 2).contiguous()
#
# decode_latent_and_save_audio(ref, vae, "lora")
# del vae
# torch.cuda.empty_cache()
# gc.collect()

In [3]:
vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device, model_dtype)
path = "inputs/lora"
load_batch_size = 2  # due to VRAM constraints, can only load in limited batch sizes
train_data_limit = 30
files = get_files_in_path_as_array(path)
wavs = load_finetuning_audio_segments(files, device, model_dtype)
y = torch.Tensor().to(device).to(model_dtype)
total = min(len(wavs), train_data_limit)
with torch.no_grad():
    for start in tqdm(range(0, total, load_batch_size)):
        end = min(start + load_batch_size, total)
        batch = encode_audio_segments(vae, wavs[start:end], device, model_dtype)
        y = torch.cat((y, batch), 0)
        del batch
        torch.cuda.empty_cache()
        gc.collect()
del vae

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention


W0609 22:21:50.914000 26068 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
100%|██████████| 12/12 [26:03<00:00, 130.32s/it]


In [4]:
class AudioLatentDataset(Dataset):
    def __init__(self, audio_latents):
        self.audio_latents = audio_latents

    def __len__(self):
        return self.audio_latents.shape[0]

    def __getitem__(self, item):
        return self.audio_latents[item]

n = y.shape[0]
perm = torch.randperm(n)
y_shuffled = y[perm]

split = int(0.8 * n)
train_dataset = AudioLatentDataset(y_shuffled[:split])
test_dataset = AudioLatentDataset(y_shuffled[split:])

In [5]:
batch_size = 1
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=True)

In [3]:
dit = load_transformer_model(model_repo, model_dtype, device)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:454: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\vector_quantize_pytorch.py:639: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\finite_scalar_quantization.py:159: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\lookup_free_quantization.py:244: FutureWarning: `torch.cuda.amp.autocast(args...)

In [4]:
for parameter in dit.parameters():
    parameter.requires_grad = False

In [5]:
import math

rank = 8
lora_alpha = rank
train_from_layer_idx = int(len(dit.decoder.layers) / 2)
for layer in dit.decoder.layers[train_from_layer_idx:]:
    layer.self_attn.q_proj.q_A = nn.Parameter(
        torch.randn(rank, layer.self_attn.q_proj.in_features, device=device, dtype=model_dtype) / math.sqrt(rank))
    layer.self_attn.q_proj.q_B = nn.Parameter(
        torch.zeros(layer.self_attn.q_proj.out_features, rank, device=device, dtype=model_dtype))

    layer.self_attn.k_proj.k_A = nn.Parameter(
        torch.randn(rank, layer.self_attn.k_proj.in_features, device=device, dtype=model_dtype) / math.sqrt(rank))
    layer.self_attn.k_proj.k_B = nn.Parameter(
        torch.zeros(layer.self_attn.k_proj.out_features, rank, device=device, dtype=model_dtype))

    layer.self_attn.v_proj.v_A = nn.Parameter(
        torch.randn(rank, layer.self_attn.v_proj.in_features, device=device, dtype=model_dtype) / math.sqrt(rank))
    layer.self_attn.v_proj.v_B = nn.Parameter(
        torch.zeros(layer.self_attn.v_proj.out_features, rank, device=device, dtype=model_dtype))

    layer.self_attn.o_proj.o_A = nn.Parameter(
        torch.randn(rank, layer.self_attn.o_proj.in_features, device=device, dtype=model_dtype) / math.sqrt(rank))
    layer.self_attn.o_proj.o_B = nn.Parameter(
        torch.zeros(layer.self_attn.o_proj.out_features, rank, device=device, dtype=model_dtype))

In [6]:
lora_scale = lora_alpha / rank

def lora_q_forward_hook(module, inputs, outputs):
    x = inputs[0]
    dW = torch.matmul(module.q_B, module.q_A)
    return outputs + lora_scale * torch.matmul(x, dW.T)

def lora_k_forward_hook(module, inputs, outputs):
    x = inputs[0]
    dW = torch.matmul(module.k_B, module.k_A)
    return outputs + lora_scale * torch.matmul(x, dW.T)

def lora_v_forward_hook(module, inputs, outputs):
    x = inputs[0]
    dW = torch.matmul(module.v_B, module.v_A)
    return outputs + lora_scale * torch.matmul(x, dW.T)

def lora_o_forward_hook(module, inputs, outputs):
    x = inputs[0]
    dW = torch.matmul(module.o_B, module.o_A)
    return outputs + lora_scale * torch.matmul(x, dW.T)

hooks = []
for layer in dit.decoder.layers[train_from_layer_idx:]:
    q_hook = layer.self_attn.q_proj.register_forward_hook(lora_q_forward_hook)
    k_hook = layer.self_attn.k_proj.register_forward_hook(lora_k_forward_hook)
    v_hook = layer.self_attn.v_proj.register_forward_hook(lora_v_forward_hook)
    o_hook = layer.self_attn.o_proj.register_forward_hook(lora_o_forward_hook)
    hooks.append(q_hook)
    hooks.append(k_hook)
    hooks.append(v_hook)
    hooks.append(o_hook)

In [7]:
# Run this to load checkpoint
dit.decoder.load_state_dict(torch.load(checkpoint_file, weights_only=True))

<All keys matched successfully>

In [8]:
silence_latent = load_silence_latent(model_repo, "silence_latent.pt", device, model_dtype)

In [9]:
def do_inference(model, batch_size=1):
    text_hidden_states = torch.zeros(batch_size, 77, 1024, dtype=model_dtype, device=device)
    text_attention_mask = torch.zeros(text_hidden_states.shape[0], text_hidden_states.shape[1], dtype=torch.bool,
                                      device=device)
    lyric_hidden_states = torch.zeros(batch_size, 123, 1024, dtype=model_dtype, device=device)
    lyric_attention_mask = torch.zeros(batch_size, 123, dtype=torch.bool, device=device)

    is_covers = torch.Tensor([False]).repeat(batch_size).to(device)

    seconds = 60
    infer_steps = 50
    frames_per_second = 25

    seq_len = int(seconds * frames_per_second)

    refer_audio_acoustic_hidden_states_packed = silence_latent[:, :, :1500].repeat(batch_size, 1, 1).permute(0, 2, 1)
    refer_audio_order_mask = torch.LongTensor(range(0, batch_size)).to(device)

    cur_chunk_mask = torch.ones(batch_size, seq_len, 64, dtype=torch.bool, device=device)
    cur_src_latents = silence_latent[:, :, :seq_len].repeat(batch_size, 1, 1).permute(0, 2, 1)

    outputs = model.generate_audio(
        text_hidden_states=text_hidden_states,
        text_attention_mask=text_attention_mask,
        lyric_hidden_states=lyric_hidden_states,
        lyric_attention_mask=lyric_attention_mask,
        refer_audio_acoustic_hidden_states_packed=refer_audio_acoustic_hidden_states_packed,
        refer_audio_order_mask=refer_audio_order_mask,
        src_latents=cur_src_latents,
        chunk_masks=cur_chunk_mask,
        infer_steps=infer_steps,
        is_covers=is_covers,
        silence_latent=silence_latent,
        use_progress_bar=True,
        shift=1.0,

        repainting_start=torch.tensor([1.0]),
        repainting_end=torch.tensor([0.0]),
        audio_cover_strength=1.0,
        use_repainting=False
    )
    output_latents = outputs['target_latents'].transpose(1, 2).contiguous()
    return output_latents

In [10]:
def do_prediction(model, xt, t_curr_tensor, params):

    decoder_outputs = model.decoder(
        hidden_states=xt,
        timestep=t_curr_tensor,
        timestep_r=t_curr_tensor,
        attention_mask=params['attention_mask'],
        encoder_hidden_states=params['encoder_hidden_states'],
        encoder_attention_mask=params['encoder_attention_mask'],
        context_latents=params['context_latents'],
        use_cache=True,
        past_key_values=None,
    )

    return decoder_outputs[0]

In [11]:
def prepare_inference(model, silence_latent, batch_size, device, model_dtype):

    with torch.no_grad():
        text_hidden_states = torch.zeros(batch_size, 77, 1024, dtype=model_dtype, device=device)
        text_attention_mask = torch.zeros(text_hidden_states.shape[0], text_hidden_states.shape[1], dtype=torch.bool,
                                          device=device)
        lyric_hidden_states = torch.zeros(batch_size, 123, 1024, dtype=model_dtype, device=device)
        lyric_attention_mask = torch.zeros(batch_size, 123, dtype=torch.bool, device=device)

        is_covers = torch.Tensor([False]).repeat(batch_size).to(device)

        seconds = 60
        frames_per_second = 25

        seq_len = int(seconds * frames_per_second)

        refer_audio_acoustic_hidden_states_packed = silence_latent[:, :, :1500].repeat(batch_size, 1, 1).permute(0, 2, 1)
        refer_audio_order_mask = torch.LongTensor(range(0, batch_size)).to(device)

        cur_chunk_mask = torch.ones(batch_size, seq_len, 64, dtype=torch.bool, device=device)
        cur_src_latents = silence_latent[:, :, :seq_len].repeat(batch_size, 1, 1).permute(0, 2, 1)

        attention_mask = torch.ones(cur_src_latents.shape[0], cur_src_latents.shape[1], device=device, dtype=model_dtype)

        encoder_hidden_states, encoder_attention_mask, context_latents = model.prepare_condition(
            text_hidden_states=text_hidden_states,
            text_attention_mask=text_attention_mask,
            lyric_hidden_states=lyric_hidden_states,
            lyric_attention_mask=lyric_attention_mask,
            refer_audio_acoustic_hidden_states_packed=refer_audio_acoustic_hidden_states_packed,
            refer_audio_order_mask=refer_audio_order_mask,
            hidden_states=cur_src_latents,
            attention_mask=attention_mask,
            silence_latent=silence_latent,
            src_latents=cur_src_latents,
            chunk_masks=cur_chunk_mask,
            is_covers=is_covers,
            precomputed_lm_hints_25Hz=None,
            audio_codes=None,
        )

        bsz = context_latents.shape[0]

        noise = model.prepare_noise(context_latents, None)

    return {'attention_mask': attention_mask, 'encoder_hidden_states': encoder_hidden_states,
            'encoder_attention_mask': encoder_attention_mask, 'context_latents': context_latents, 'noise': noise,
            'bsz': bsz}

In [12]:
def train_loop(dataloader, model, loss_fn, optimizer):
    timesteps = [1.0, 0.875, 0.75, 0.625, 0.5, 0.375, 0.25, 0.125]
    size = len(dataloader.dataset)

    for batch, y in enumerate(dataloader):
        t_idx = random.randrange(len(timesteps))
        current_timestep = timesteps[t_idx]
        t_curr_tensor = current_timestep * torch.ones((batch_size,), device=device, dtype=model_dtype)

        params = prepare_inference(model, silence_latent, batch_size, device, model_dtype)
        noise = params["noise"]

        t_ = t_curr_tensor.unsqueeze(-1).unsqueeze(-1)
        xt = noise * t_ + (1 - t_) * y.permute(0, 2, 1)

        pred_noise = do_prediction(model, xt, t_curr_tensor, params)

        target_velocity = noise - y.permute(0, 2, 1)

        loss = loss_fn(pred_noise, target_velocity)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in dit.parameters() if p.requires_grad], max_norm=1.0)
        optimizer.step()
        optimizer.zero_grad()

        loss, current = loss.item(), batch * batch_size
        layer = dit.decoder.layers[train_from_layer_idx]
        norm = torch.matmul(layer.self_attn.q_proj.q_B, layer.self_attn.q_proj.q_A).norm().item()
        print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]  q_dW_norm: {norm:.4f}")

def test_loop(dataloader, model, loss_fn):
    timesteps = [1.0, 0.875, 0.75, 0.625, 0.5, 0.375, 0.25, 0.125]
    num_batches = len(dataloader)
    test_loss = 0

    with torch.no_grad():
        for y in dataloader:
            t_idx = random.randrange(len(timesteps))
            current_timestep = timesteps[t_idx]
            t_curr_tensor = current_timestep * torch.ones((batch_size,), device=device, dtype=model_dtype)

            params = prepare_inference(model, silence_latent, batch_size, device, model_dtype)
            noise = params["noise"]

            t_ = t_curr_tensor.unsqueeze(-1).unsqueeze(-1)
            xt = noise * t_ + (1 - t_) * y.permute(0, 2, 1)

            pred_noise = do_prediction(model, xt, t_curr_tensor, params)

            target_velocity = noise - y.permute(0, 2, 1)

            test_loss += loss_fn(pred_noise, target_velocity).item()

    test_loss /= num_batches
    print(f"Avg Test loss: {test_loss:>8f} \n")

In [13]:
def decode_and_save(output_latents):
    vae = load_encoder("./models/ace-step-vae/config.json", "./models/ace-step-vae/checkpoint.ckpt", device,
                       model_dtype)
    with torch.no_grad():
        decode_latent_and_save_audio(output_latents, vae, "lora")
    del vae
    torch.cuda.empty_cache()
    gc.collect()

In [16]:
loss_fn = nn.MSELoss()
lora_params = [p for p in dit.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(lora_params, lr=1e-4, weight_decay=0.01)

epochs = 30
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    train_loop(train_dataloader, dit, loss_fn, optimizer)
    test_loop(test_dataloader, dit, loss_fn)

Epoch 1
-------------------------------


C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\residual_fsq.py:170: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled = False):


loss: 1.945312  [    0/   19]  q_dW_norm: 0.2051
loss: 3.406250  [    1/   19]  q_dW_norm: 0.3184
loss: 1.640625  [    2/   19]  q_dW_norm: 0.3965
loss: 1.687500  [    3/   19]  q_dW_norm: 0.4766
loss: 1.375000  [    4/   19]  q_dW_norm: 0.5430
loss: 0.960938  [    5/   19]  q_dW_norm: 0.5977
loss: 1.000000  [    6/   19]  q_dW_norm: 0.6484
loss: 1.414062  [    7/   19]  q_dW_norm: 0.7031
loss: 0.996094  [    8/   19]  q_dW_norm: 0.7578
loss: 1.421875  [    9/   19]  q_dW_norm: 0.8086
loss: 1.273438  [   10/   19]  q_dW_norm: 0.8594
loss: 1.304688  [   11/   19]  q_dW_norm: 0.9102
loss: 1.093750  [   12/   19]  q_dW_norm: 0.9570
loss: 1.726562  [   13/   19]  q_dW_norm: 1.0078
loss: 1.015625  [   14/   19]  q_dW_norm: 1.0625
loss: 1.359375  [   15/   19]  q_dW_norm: 1.1094
loss: 1.296875  [   16/   19]  q_dW_norm: 1.1562
loss: 1.015625  [   17/   19]  q_dW_norm: 1.2109
loss: 1.328125  [   18/   19]  q_dW_norm: 1.2578
Avg Test loss: 1.340625 

Epoch 2
-------------------------------
los

In [17]:
torch.save(dit.decoder.state_dict(), checkpoint_file)

In [38]:
if hooks:
    for hook in hooks:
        hook.remove()

In [ ]:
with torch.no_grad():
    for layer in dit.decoder.layers[train_from_layer_idx:]:
        layer.self_attn.q_proj.weight.add_(lora_scale * torch.matmul(layer.self_attn.q_proj.q_B, layer.self_attn.q_proj.q_A))
        layer.self_attn.k_proj.weight.add_(lora_scale * torch.matmul(layer.self_attn.k_proj.k_B, layer.self_attn.k_proj.k_A))
        layer.self_attn.v_proj.weight.add_(lora_scale * torch.matmul(layer.self_attn.v_proj.v_B, layer.self_attn.v_proj.v_A))
        layer.self_attn.o_proj.weight.add_(lora_scale * torch.matmul(layer.self_attn.o_proj.o_B, layer.self_attn.o_proj.o_A))

In [14]:
output_latents = do_inference(dit, 1)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\vector_quantize_pytorch\residual_fsq.py:170: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled = False):


In [15]:
del dit
torch.cuda.empty_cache()
gc.collect()

1543

In [16]:
decode_and_save(output_latents)

C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\clip\clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention


W0610 17:23:38.942000 22292 .venv\Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
C:\repos\python\audio-diffusion-finetuning\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


torch.Size([1, 2, 2880000])
Audio saved to outputs\lora0.wav
(2880000, 2)
